# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/khanarmaghanrasheed-18/FlyRankAi-KhanArmaghan-Internship-2026/blob/main/work/notebooks/w03_data_contract.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

One row = one pseudonymized content page. The available metrics are aggregated over a trailing 90-day window, with additional 30-day comparison fields for trend inputs.

Verify the row grain, the client grouping, and the measurement windows in the code cell below.


In [1]:
from pathlib import Path
import pandas as pd

data_path = Path("../../data/raw/content_refresh_anonymized.csv")
if not data_path.exists():
    data_path = Path("data/raw/content_refresh_anonymized.csv")

df = pd.read_csv(data_path)

unit_checks = {
    "total_rows": len(df),
    "unique_content_id": df["content_id"].nunique(),
    "unique_client_id": df["client_id"].nunique(),
    "has_90d_fields": all(col in df.columns for col in [
        "impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d",
        "users_90d", "engaged_sessions_90d", "ai_sessions_90d", "scroll_events_90d",
    ]),
    "has_30d_trend_fields": all(col in df.columns for col in [
        "impressions_last_30d", "impressions_prev_30d",
        "clicks_last_30d", "clicks_prev_30d",
        "sessions_last_30d", "sessions_prev_30d",
    ]),
}
unit_checks


{'total_rows': 30000,
 'unique_content_id': 30000,
 'unique_client_id': 32,
 'has_90d_fields': True,
 'has_30d_trend_fields': True}

## 2. Fields: feature / label / context / excluded

Features are the page-level inputs we can use for candidate ranking. Labels are not available for true cannibalization, so we treat trend fields as label-related and avoid them. Context fields are grouping-only. Excluded fields are either label/leakage, pseudonymous IDs, redundant, derived, too sparse, or not helpful for this ranking task.


In [2]:
from pathlib import Path
import pandas as pd

data_path = Path("../../data/raw/content_refresh_anonymized.csv")
if not data_path.exists():
    data_path = Path("data/raw/content_refresh_anonymized.csv")

df = pd.read_csv(data_path)

feature_columns = [
    "search_volume", "competition", "competition_level", "cpc", "main_intent",
    "content_age_days", "days_since_last_update",
    "impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d",
    "scroll_events_90d", "days_with_impressions", "days_with_sessions",
    "impressions_last_30d", "clicks_last_30d", "sessions_last_30d",
    "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d",
    "ctr", "avg_position", "engagement_rate", "scroll_rate",
]
label_columns = ["trend_direction", "trend_pct"]
context_columns = ["content_id", "client_id"]
excluded_columns = {
    "content_type": "Highly imbalanced and too constant to be a useful discriminative feature",
    "word_count": "Redundant with char_count and missing for 7,699 rows",
    "char_count": "Redundant with word_count and missing for 7,699 rows",
    "provider_used": "Mostly missing and unrelated to cannibalization risk",
    "model_used": "Too sparse and not a semantic signal for this task",
    "engaged_sessions_90d": "Redundant with sessions_90d and often zero",
    "ai_sessions_90d": "Mostly zero and not generally useful for content competition",
    "ai_traffic_pct": "Mostly zero and derived from ai_sessions_90d",
    "users_90d": "Almost perfectly correlated with sessions_90d",
    "age_tier": "Derived from content_age_days",
    "age_tier_order": "Derived from content_age_days",
    "freshness_tier": "Derived from days_since_last_update",
    "word_count_tier": "Derived from word_count and missing where word_count is missing",
    "char_count_tier": "Derived from char_count and missing where char_count is missing",
    "impression_tier": "Derived from impressions_90d",
    "position_tier": "Derived from avg_position",
    "trend_direction": "Label source, not a feature",
    "trend_pct": "Label source, not a feature",
}
pd.DataFrame([
    {"category": "feature", "column": col} for col in feature_columns
] + [
    {"category": "label", "column": col} for col in label_columns
] + [
    {"category": "context", "column": col} for col in context_columns
] + [
    {"category": "excluded", "column": col, "reason": reason} for col, reason in excluded_columns.items()
])


,category,column,reason
0,feature,search_volume,NaN
1,feature,competition,NaN
2,feature,competition_level,NaN
3,feature,cpc,NaN
4,feature,main_intent,NaN
5,feature,content_age_days,NaN
6,feature,days_since_last_update,NaN
7,feature,impressions_90d,NaN
8,feature,clicks_90d,NaN
9,feature,pageviews_90d,NaN


## 3. Verify it with queries (grain, counts, missing values, windows)

Use real data checks to verify the row grain, confirm the systematic missingness of excluded fields, and prove the redundancy or imbalance claims for derived values.


In [3]:
from pathlib import Path
import pandas as pd

data_path = Path("../../data/raw/content_refresh_anonymized.csv")
if not data_path.exists():
    data_path = Path("data/raw/content_refresh_anonymized.csv")

df = pd.read_csv(data_path)

verification = {
    "row_grain_matches_ids": len(df) == df["content_id"].nunique(),
    "client_count": df["client_id"].nunique(),
    "keyword_article_share": (df["content_type"] == "keyword article").mean(),
    "word_count_missing": df["word_count"].isna().sum(),
    "char_count_missing": df["char_count"].isna().sum(),
    "word_char_corr": df[["word_count","char_count"]].corr().iloc[0,1],
    "provider_used_missing": df["provider_used"].isna().sum(),
    "provider_used_google_share": (df["provider_used"] == "google").mean(),
    "engaged_sessions_zero_share": (df["engaged_sessions_90d"] == 0).mean(),
    "ai_sessions_zero_share": (df["ai_sessions_90d"] == 0).mean(),
    "ai_traffic_zero_share": (df["ai_traffic_pct"] == 0).mean(),
    "trend_direction_present": "trend_direction" in df.columns,
    "trend_pct_missing": df["trend_pct"].isna().sum(),
    "users_sessions_corr": df[["users_90d","sessions_90d"]].corr().iloc[0,1],
    "derived_buckets_present": all(col in df.columns for col in [
        "age_tier", "age_tier_order", "freshness_tier",
        "word_count_tier", "char_count_tier", "impression_tier", "position_tier"
    ]),
}
pd.Series(verification, name="verification")


row_grain_matches_ids              True
client_count                         32
keyword_article_share            0.9069
word_count_missing                 7699
char_count_missing                 7699
word_char_corr                 0.939252
provider_used_missing             21438
provider_used_google_share     0.245467
engaged_sessions_zero_share    0.720967
ai_sessions_zero_share         0.935667
ai_traffic_zero_share          0.935667
trend_direction_present            True
trend_pct_missing                  3388
users_sessions_corr            0.998111
derived_buckets_present            True
Name: verification, dtype: object

## 4. Data limits

This data can support an unsupervised ranking system, but it cannot prove true cannibalization. The limits below describe what the slice cannot tell us and why the project must stay in decision-support mode.


In [4]:
limitations = [
    "Can only support candidate ranking, not ground-truth cannibalization labels.",
    "Trend fields (trend_direction, trend_pct) are label sources and must not be used as features.",
    "The 90-day and 30-day aggregates are page-level summaries, not query-level overlap measures.",
    "content_id and client_id are pseudonyms for grouping only and do not carry semantic meaning.",
    "Some columns are too sparse or redundant for this task, such as provider_used, model_used, ai_sessions_90d, ai_traffic_pct, and users_90d.",
    "Derived buckets like age_tier, word_count_tier, impression_tier, and position_tier are redundant when the underlying numeric fields are available.",
]
limitations


['Can only support candidate ranking, not ground-truth cannibalization labels.',
 'Trend fields (trend_direction, trend_pct) are label sources and must not be used as features.',
 'The 90-day and 30-day aggregates are page-level summaries, not query-level overlap measures.',
 'content_id and client_id are pseudonyms for grouping only and do not carry semantic meaning.',
 'Some columns are too sparse or redundant for this task, such as provider_used, model_used, ai_sessions_90d, ai_traffic_pct, and users_90d.',
 'Derived buckets like age_tier, word_count_tier, impression_tier, and position_tier are redundant when the underlying numeric fields are available.']

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.